# H2 — GDD e volume de irrigação por evento

Hipótese: o GDD (Growing Degree Days) acumulado está associado ao volume de água aplicado em cada evento de irrigação. O volume é calculado pela diferença do medidor durante o evento. Como o GDD já está no dataset processado e é um indicador agronômico diário, sua associação usa somente o timestamp da abertura, sem incluir a linha no casamento.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'src').exists():
    ROOT = (Path.cwd() / '..').resolve()
sys.path.insert(0, str(ROOT))

from src.utils import extrair_eventos_irrigacao

In [2]:
def recupera_gdd(df, eventos):
    # Recupera o GDD já existente no timestamp da abertura.
    dados = df[['timestamp_10min', 'gdd']].copy()
    dados['timestamp_10min'] = pd.to_datetime(
        dados['timestamp_10min'], errors='coerce', utc=True
    )
    dados['gdd'] = pd.to_numeric(dados['gdd'], errors='coerce')
    dados = (
        dados.dropna(subset=['timestamp_10min'])
        .drop_duplicates(subset=['timestamp_10min'])
        .rename(columns={'timestamp_10min': 'timestamp_abertura', 'gdd': 'gdd_evento'})
    )
    return eventos.merge(dados, on='timestamp_abertura', how='left')


def plot_eventos_gdd(eventos):
    # Gera o scatter de GDD diário e volume do evento.
    dados = eventos.dropna(subset=['gdd_evento', 'volume_evento']).copy()
    if dados.empty:
        raise ValueError('Não há eventos válidos para plotar')

    saida = ROOT / 'imgs' / 'H2' / 'H2_eventos_gdd_vs_volume.png'
    saida.parent.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.scatter(
        dados['gdd_evento'], dados['volume_evento'],
        color='#2171b5', edgecolors='#08306b', linewidths=0.4, alpha=0.8,
    )
    ax.set_xlabel('GDD acumulado no dia da abertura (°C·dia)')
    ax.set_ylabel('Volume de água no evento (m³)')
    ax.grid(color='#c6dbef', alpha=0.45)
    fig.tight_layout()
    fig.savefig(saida, dpi=150, bbox_inches='tight')
    plt.close(fig)
    return saida

In [3]:
# Mesma seleção da análise principal do H1: safra 2025, linha 5.
df_h2 = pd.read_csv(
    ROOT / 'data' / 'processed' / 'dataset_m2_2025.csv',
    low_memory=False,
)
df_h2 = df_h2.loc[df_h2['line'].eq(4)].copy()
df_h2['safra'] = '2024'

In [4]:
# Eventos: transição 0→1, volume do evento e GDD do dia da abertura.
timestamps_monitoramento = pd.to_datetime(
    df_h2['timestamp_10min'], errors='coerce', utc=True
).dropna()
eventos_h2 = recupera_gdd(
    df_h2, extrair_eventos_irrigacao(df_h2)
)
figura_scatter = plot_eventos_gdd(eventos_h2)
print(f'Início do monitoramento: {timestamps_monitoramento.min():%d/%m/%Y}')
print(f'Fim do monitoramento: {timestamps_monitoramento.max():%d/%m/%Y}')
print(f'Figura salva em: {figura_scatter}')

Início do monitoramento: 26/06/2025
Fim do monitoramento: 28/10/2025
Figura salva em: /mnt/c/Users/ander/Desktop/Faculdade/PI4comIOT/projeto-integrador-iv-mitha/imgs/H2/H2_eventos_gdd_vs_volume.png


In [5]:
# Correlação por evento: GDD diário × volume do evento.
rho = eventos_h2['gdd_evento'].corr(
    eventos_h2['volume_evento'], method='spearman'
)
print(f'Correlação de Spearman por evento: {rho:.3f}')

Correlação de Spearman por evento: 0.135


In [6]:
# Tabela comparativa: mesmas seleções do bloco comparativo do H1.
tabelas_eventos = []
for safra in ('2024', '2025'):
    dados_safra = pd.read_csv(
        ROOT / 'data' / 'processed' / f'dataset_m2_{safra}.csv',
        low_memory=False,
    )
    linhas = [4] if safra == '2024' else [1]
    for linha in linhas:
        dados_linha = dados_safra.loc[dados_safra['line'].eq(linha)].copy()
        eventos_linha = recupera_gdd(
            dados_linha, extrair_eventos_irrigacao(dados_linha)
        )
        eventos_linha.insert(0, 'line', linha)
        eventos_linha.insert(0, 'safra', safra)
        tabelas_eventos.append(eventos_linha)

tabela_linha = pd.concat(tabelas_eventos, ignore_index=True)
tabela_linha = tabela_linha[[
    'safra', 'line', 'evento', 'timestamp_pre_abertura',
    'timestamp_abertura', 'gdd_evento', 'volume_evento'
]]
print(tabela_linha.to_string(index=False))

safra  line  evento    timestamp_pre_abertura        timestamp_abertura  gdd_evento  volume_evento
 2024     4       1 2024-07-26 06:50:00+00:00 2024-07-26 07:00:00+00:00      1130.6         1607.0
 2024     4       2 2024-07-27 06:50:00+00:00 2024-07-27 07:00:00+00:00      1150.4         1461.0
 2024     4       3 2024-07-29 06:50:00+00:00 2024-07-29 07:00:00+00:00      1191.8         1725.0
 2024     4       4 2024-07-30 06:50:00+00:00 2024-07-30 07:00:00+00:00      1211.4         1636.0
 2024     4       5 2024-08-02 07:00:00+00:00 2024-08-02 07:10:00+00:00      1273.4         1579.0
 2024     4       6 2024-08-03 07:00:00+00:00 2024-08-03 07:10:00+00:00      1292.6         1276.0
 2024     4       7 2024-08-06 07:00:00+00:00 2024-08-06 07:10:00+00:00      1348.5         1661.0
 2024     4       8 2024-08-08 07:00:00+00:00 2024-08-08 07:10:00+00:00      1383.0         1357.0
 2024     4       9 2024-08-15 06:50:00+00:00 2024-08-15 07:00:00+00:00      1502.0         1453.0
 2024     

In [7]:
# Boxplots por safra, agrupando os eventos em tercis de GDD.
# Mantém os mesmos fatores de normalização do H1.
regime_irriframe = {
    ('2024', 4): 1.00,
    ('2025', 1): 1.00,
}
dados_boxplot = tabela_linha.copy()
dados_boxplot['fator_irriframe'] = [
    regime_irriframe[(str(safra), int(linha))]
    for safra, linha in zip(dados_boxplot['safra'], dados_boxplot['line'])
]
dados_boxplot['volume_equivalente_100'] = (
    dados_boxplot['volume_evento'] / dados_boxplot['fator_irriframe']
)

for safra in ('2024', '2025'):
    dados_ano = dados_boxplot.loc[dados_boxplot['safra'].eq(safra)].copy()
    dados_ano['faixa_gdd'] = pd.qcut(
        dados_ano['gdd_evento'], q=3, duplicates='drop'
    )

    grupos = []
    rotulos = []
    for faixa in dados_ano['faixa_gdd'].cat.categories:
        valores = dados_ano.loc[
            dados_ano['faixa_gdd'].eq(faixa), 'volume_equivalente_100'
        ].dropna()
        if valores.empty:
            continue
        gdd = dados_ano.loc[
            dados_ano['faixa_gdd'].eq(faixa), 'gdd_evento'
        ]
        grupos.append(valores.to_numpy())
        rotulos.append(f'{gdd.min():.1f}–{gdd.max():.1f}')

    fig, ax = plt.subplots(figsize=(9, 6))
    boxplot = ax.boxplot(grupos, tick_labels=rotulos, patch_artist=True)
    for caixa, cor in zip(boxplot['boxes'], ('#c6dbef', '#6baed6', '#2171b5')):
        caixa.set_facecolor(cor)
    linha_safra = next(
        linha
        for (safra_dict, linha), fator in regime_irriframe.items()
        if safra_dict == safra and fator == 1.00
    )
    ax.set_title(f'Volume equivalente na linha {linha_safra} — {safra}')
    ax.set_xlabel('GDD acumulado no dia da abertura (°C·dia)')
    ax.set_ylabel('Volume (m³)')
    ax.grid(axis='y', color='#c6dbef', alpha=0.45)
    fig.tight_layout()
    saida = ROOT / 'imgs' / 'H2' / f'H2_boxplot_volume_equivalente_100_{safra}.png'
    saida.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(saida, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'Figura salva em: {saida}')

Figura salva em: /mnt/c/Users/ander/Desktop/Faculdade/PI4comIOT/projeto-integrador-iv-mitha/imgs/H2/H2_boxplot_volume_equivalente_100_2024.png
Figura salva em: /mnt/c/Users/ander/Desktop/Faculdade/PI4comIOT/projeto-integrador-iv-mitha/imgs/H2/H2_boxplot_volume_equivalente_100_2025.png


In [8]:
# Scatter comparativo: eventos de mais de uma safra.
dados_scatter = tabela_linha.dropna(
    subset=['gdd_evento', 'volume_evento']
).copy()
cores_safra = {'2024': '#2171b5', '2025': '#d73027'}

fig, ax = plt.subplots(figsize=(9, 6))
for safra, dados_safra in dados_scatter.groupby('safra', sort=True):
    linhas = ', '.join(
        str(int(linha)) for linha in sorted(dados_safra['line'].dropna().unique())
    )
    ax.scatter(
        dados_safra['gdd_evento'], dados_safra['volume_evento'],
        color=cores_safra.get(safra, '#636363'),
        edgecolors='#252525', linewidths=0.4, alpha=0.8,
        label=f'Safra {safra} — linha(s) {linhas}',
    )

ax.set_title('GDD e volume de irrigação por evento de acionamento')
ax.set_xlabel('GDD acumulado no dia da abertura (°C·dia)')
ax.set_ylabel('Volume de água no evento (m³)')
ax.legend(frameon=False)
ax.grid(color='#c6dbef', alpha=0.45)
fig.tight_layout()
saida = ROOT / 'imgs' / 'H2' / 'H2_eventos_gdd_vs_volume_por_safra.png'
saida.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(saida, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Figura salva em: {saida}')

Figura salva em: /mnt/c/Users/ander/Desktop/Faculdade/PI4comIOT/projeto-integrador-iv-mitha/imgs/H2/H2_eventos_gdd_vs_volume_por_safra.png
